# Baseline vs advanced RAG

Interview demo for Phase 8. Same 30 gold questions from `data/eval/rag_eval.jsonl` (Gao et al. RAG survey PDF), two graphs:

| Pipeline | Graph |
|----------|--------|
| **Baseline** | vector search → generate (Phase 1) |
| **Advanced** | hybrid RRF → rerank → prune → quality/rewrite → generate |

Metrics: retrieval (hit rate, recall, precision, MRR), context (pages actually sent to the LLM), generation (faithfulness, relevance), system (latency, tokens).

Ingest first (`python -m app.cli ingest`). Then either run this notebook with `RUN_LIVE = True`, or run `python -m app.cli eval` and load `data/eval/last_benchmark.json`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.eval.dataset import load_eval_dataset
from app.eval.evaluators import format_comparison_table
from app.eval.benchmark import DEFAULT_RESULTS_PATH, load_benchmark, run_benchmark, save_benchmark

examples = load_eval_dataset()
print(f"{len(examples)} gold questions")
for row in examples[:5]:
    print(f"  {row['id']}: {row['question']}  pages={row['expected_pages']}")

## How to read the table

- **hit_rate / recall / precision / mrr** — ranked retrieved pages vs `expected_pages`. Advanced should win on exact terms (DenseX, FLARE) because BM25 + RRF + rerank promote the table row.
- **context_hit_rate / context_recall** — pages in the prompt after prune. Advanced sends ~5 chunks; baseline dumps the full vector top-k. If context recall stays high while **context_tokens** drop, pruning kept the gold page.
- **faithfulness** — answer tokens that appear in retrieved context (grounding, not LLM-as-judge).
- **relevance** — token F1 vs `reference_answer`.
- **latency_ms** — advanced can be slower (rerank + possible rewrite) while using fewer prompt tokens.

In [ ]:
RUN_LIVE = False  # True calls OpenAI twice per question
LIMIT = 5         # used only when RUN_LIVE

if RUN_LIVE:
    subset = examples[:LIMIT] if LIMIT else examples
    result = run_benchmark(subset, pipelines=("baseline", "advanced"))
    save_benchmark(result)
    print(f"scored {result['n']} questions")
elif DEFAULT_RESULTS_PATH.is_file():
    result = load_benchmark()
    print(f"loaded {DEFAULT_RESULTS_PATH} ({result['n']} questions)")
else:
    result = None
    print("No results yet. Run: python -m app.cli eval --limit 5")
    print("Or set RUN_LIVE = True in this cell.")

In [ ]:
if result and result.get("comparison"):
    print(format_comparison_table(result["comparison"]))
    print()
    for name, payload in result["per_pipeline"].items():
        summary = payload["summary"]
        print(
            f"{name}: hit={summary['hit_rate']:.2f}  "
            f"mrr={summary['mrr']:.2f}  "
            f"ctx_recall={summary['context_recall']:.2f}  "
            f"faith={summary['faithfulness']:.2f}  "
            f"rel={summary['relevance']:.2f}  "
            f"{summary['latency_ms']:.0f}ms  "
            f"{summary['context_tokens']:.0f} ctx tokens"
        )
else:
    print("Run the previous cell first.")

In [ ]:
if result:
    try:
        import pandas as pd

        display(pd.DataFrame(result["comparison"]))
    except ImportError:
        pass

    print("\nPer-question context hit (advanced vs baseline)")
    base_rows = result["per_pipeline"]["baseline"]["rows"]
    adv_rows = result["per_pipeline"]["advanced"]["rows"]
    by_id = {row["id"]: row for row in base_rows}
    for adv in adv_rows[:12]:
        base = by_id.get(adv["id"], {})
        print(
            f"{adv['id']}: base_ctx={base.get('context_hit_rate', 0):.0f} "
            f"adv_ctx={adv['context_hit_rate']:.0f}  {adv['question'][:70]}"
        )

## Upload to LangSmith (optional)

```python
from app.eval.dataset import upload_langsmith_dataset
upload_langsmith_dataset(examples, overwrite=False)
```

Needs `LANGSMITH_API_KEY`. Dataset name: `rag-eval`. Each example is `{inputs: question, outputs: reference_answer, expected_pages}`.

CLI equivalent: `python -m app.cli eval --upload`.